# Robust Accelerometer Pipeline with Hyperparameter Tuning

This notebook enhances the previous accelerometer data processing pipeline by incorporating hyperparameter optimization using the `optuna` library. The goal is to find the best combination of data preprocessing parameters (window size, purity threshold) and model hyperparameters for several deep learning models.

In [1]:
import os
import pandas as pd 
import tempfile 
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score, f1_score
from scipy.signal import butter, filtfilt
import matplotlib.pyplot as plt
import seaborn as sns
import copy
import warnings
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

warnings.filterwarnings("ignore")

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu")
print(f"Using device: {device}")

/home/rajesh/work/basic_stat/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## 1. Data Loading and Initial Processing
First, we load the accelerometer data from the zip files, concatenate them, and perform a simple cleaning of the behavioral labels.

In [2]:
actual_file = []
main_path = "/home/rajesh/work/acclerometer_project/zip_data"

if os.path.exists(main_path):
    for zip_file in os.listdir(main_path):
        if zip_file.endswith(".zip"):
            with tempfile.TemporaryDirectory() as temp_dir:
                try:
                    with zipfile.ZipFile(os.path.join(main_path, zip_file), "r") as zf:
                        zf.extractall(temp_dir)
                        for root, dirs, files in os.walk(temp_dir):
                            for d in dirs:
                                if d.startswith('Processed'):
                                    sec_path = os.path.join(root, d)
                                    for f in os.listdir(sec_path):
                                        if f.endswith((".xls", ".xlsx")):
                                            actual_file.append(pd.read_excel(os.path.join(sec_path, f)))
                except Exception as e: print(f"Error: {e}")
    if actual_file: 
        df_raw = pd.concat(actual_file)
        df_raw['behavioral_category']= df_raw['behavioral_category'].apply(lambda x : 'Resting' if x=='Resting Ruminating' else x)
        print("Data loaded and cleaned successfully.")
        print("Unique behaviors:", df_raw['behavioral_category'].unique())
    else:
        print("No data loaded. Please check the path.")

Data loaded and cleaned successfully.
Unique behaviors: ['Resting' 'Walking' 'Grazing']


## 2. Pipeline, Models, and Training Functions
We define the data processing pipeline, the neural network models, and the Leave-One-Subject-Out (LOSO) training functions. A new function, `run_loso_for_tuning`, is created to accept hyperparameters, making it suitable for optimization with Optuna.

In [3]:
class AdaptiveAccelPipeline:
    def __init__(self, df):
        self.df = df.copy()
        self.df["local_ts"] = pd.to_datetime(self.df["local_ts"])
        self.df = self.df.sort_values(["subject", "local_ts"])

    def filter(self, cutoff=5.0):
        def f(g):
            dt = g["local_ts"].diff().dt.total_seconds()
            dt = dt[dt > 0]
            if len(dt) < 10:
                return g
            fs = 1 / dt.median()
            if fs <= cutoff * 2:
                return g
            nyq = fs * 0.5
            b, a = butter(4, cutoff / nyq, btype="low")
            for c in ["x", "y", "z"]:
                g[c] = filtfilt(b, a, g[c])
            return g
        self.df = self.df.groupby("subject", group_keys=False).apply(f)

    def features(self):
        scale = 16384
        self.df["x_g"] = self.df["x"] / scale
        self.df["y_g"] = self.df["y"] / scale
        self.df["z_g"] = self.df["z"] / scale
        self.df["mag"] = np.sqrt(self.df["x_g"]**2 + self.df["y_g"]**2 + self.df["z_g"]**2)
        self.df["enmo"] = np.maximum(self.df["mag"] - 1, 0)
        self.df["odba"] = ((self.df["x_g"] - self.df["x_g"] .mean()).abs() + (self.df["y_g"] - self.df["y_g"] .mean()).abs() + (self.df["z_g"] - self.df["z_g"] .mean()).abs())

    def resample(self, window=10, thresh=0.7):
        def labeler(x):
            vc = x.value_counts(normalize=True)
            if len(vc) and vc.iloc[0] >= thresh:
                return vc.index[0]
            return np.nan

        agg = {
            "x_g": ["mean", "std"], "y_g": ["mean", "std"], "z_g": ["mean", "std"],
            "mag": ["mean", "std"], "enmo": ["mean", "max"], "odba": ["mean", "std"],
            "behavioral_category": labeler
        }
        out = self.df.set_index("local_ts").groupby("subject").resample(f"{window}s").agg(agg)
        out.columns = [f"{a}_{b}" if b else a for a, b in out.columns]
        out = out.rename(columns={"behavioral_category_labeler": "label"})
        return out.dropna(subset=["label"]).reset_index()

    def sequences(self, df, feats, target, steps):
        X, y, s = [], [], []
        for sub, g in df.groupby("subject"):
            g = g.sort_values("local_ts")
            if len(g) <= steps:
                continue
            for i in range(len(g) - steps):
                X.append(g[feats].values[i:i+steps])
                y.append(g[target].iloc[i+steps])
                s.append(sub)
        return np.array(X), np.array(y), np.array(s)

# --- Models --- 
class RNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, classes):
        super().__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, classes)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, classes)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class BiLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, classes)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# --- LOSO Training Function for Optuna --- 
def run_loso_for_tuning(X, y, subjects, le, model_type, params):
    all_true, all_pred = [], []
    unique_subjects = np.unique(subjects)

    for test_subject in unique_subjects:
        tr = subjects != test_subject
        te = subjects == test_subject
        Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]

        if len(np.unique(ytr)) < 2: continue

        train_loader = DataLoader(TensorDataset(torch.tensor(Xtr).float(), torch.tensor(ytr)), batch_size=32, shuffle=True)
        test_loader = DataLoader(TensorDataset(torch.tensor(Xte).float(), torch.tensor(yte)), batch_size=32)

        input_dim = X.shape[2]
        num_classes = len(le.classes_)
        
        if model_type == "RNN": model = RNNModel(input_dim, params['hidden_dim'], num_classes)
        elif model_type == "LSTM": model = LSTMModel(input_dim, params['hidden_dim'], num_classes)
        elif model_type == "BiLSTM": model = BiLSTMModel(input_dim, params['hidden_dim'], num_classes)
        
        model.to(device)
        opt = optim.Adam(model.parameters(), lr=params['lr'])
        loss_fn = nn.CrossEntropyLoss()

        for epoch in range(15):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                opt.step()

        model.eval()
        preds = []
        with torch.no_grad():
            for xb, _ in test_loader:
                preds.extend(torch.argmax(model(xb.to(device)), 1).cpu().numpy())
        all_true.extend(yte)
        all_pred.extend(preds)

    return np.array(all_true), np.array(all_pred)

## 3. Optuna Objective Function
This function defines the search space for our hyperparameters. For each trial, Optuna selects a combination of parameters, and we run the entire data processing and training pipeline to evaluate its performance (F1-score).

In [6]:
def objective(trial, model_type):
    # 1. Suggest data processing hyperparameters
    window = trial.suggest_int("window", 10, 30, step=5)
    thresh = trial.suggest_float("thresh", 0.6, 0.8, step=0.05)
    steps = trial.suggest_int("steps", 5, 20, step=5)

    # 2. Suggest model hyperparameters
    params = {
        'hidden_dim': trial.suggest_categorical("hidden_dim", [32, 64, 128]),
        'lr': trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    }

    # 3. Run the pipeline with suggested params
    pipe = AdaptiveAccelPipeline(df_raw)
    pipe.filter()
    pipe.features()
    df_r = pipe.resample(window=window, thresh=thresh)

    if len(df_r) < 100: # Not enough data to form sequences
        return 0.0 # Prune trial

    feats = [c for c in df_r.columns if c not in ["subject", "local_ts", "label"] ]
    le = LabelEncoder()
    df_r["label_enc"] = le.fit_transform(df_r["label"])
    df_r[feats] = StandardScaler().fit_transform(df_r[feats])

    X, y, subjects = pipe.sequences(df_r, feats, "label_enc", steps=steps)

    if X.shape[0] == 0: # Check if sequences were created
        return 0.0
    
    # 4. Train and evaluate
    y_true, y_pred = run_loso_for_tuning(X, y, subjects, le, model_type, params)

    # 5. Return the F1 score for Optuna to maximize
    return f1_score(y_true, y_pred, average='macro', zero_division=0)

## 4. Hyperparameter Tuning Execution
We now run the optimization studies. We'll do this for a few models (e.g., LSTM and BiLSTM) for a set number of trials. For a real-world scenario, `n_trials` should be much higher (e.g., 50-100).

In [ ]:
models_to_tune = ["LSTM", "BiLSTM"] # Add 'RNN' if desired
studies = {}

for model_name in models_to_tune:
    print(f"\n--- Tuning {model_name} ---")
    study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
    study.optimize(lambda trial: objective(trial, model_name), n_trials=20) # Using 20 trials for demonstration
    studies[model_name] = study
    
    print(f"--- Best Trial for {model_name} ---")
    print(f"  Value: {study.best_value:.4f}")
    print("  Params: ")
    for key, value in study.best_params.items():
        print(f"    {key}: {value}")

[I 2026-01-26 17:31:17,568] A new study created in memory with name: no-name-bf3f3c3a-370b-4b5b-89f9-f99d73d347c6



--- Tuning LSTM ---


[I 2026-01-26 17:32:26,156] Trial 0 finished with value: 0.6577622435494292 and parameters: {'window': 20, 'thresh': 0.65, 'steps': 5, 'hidden_dim': 32, 'lr': 0.00568323601149098}. Best is trial 0 with value: 0.6577622435494292.
[I 2026-01-26 17:33:20,685] Trial 1 finished with value: 0.636347229410717 and parameters: {'window': 25, 'thresh': 0.8, 'steps': 5, 'hidden_dim': 128, 'lr': 0.004484152371170322}. Best is trial 0 with value: 0.6577622435494292.
[I 2026-01-26 17:34:14,494] Trial 2 finished with value: 0.6315075670228169 and parameters: {'window': 30, 'thresh': 0.7, 'steps': 10, 'hidden_dim': 64, 'lr': 0.00104371895460885}. Best is trial 0 with value: 0.6577622435494292.
[I 2026-01-26 17:37:55,759] Trial 3 finished with value: 0.6885330887142235 and parameters: {'window': 10, 'thresh': 0.6, 'steps': 15, 'hidden_dim': 32, 'lr': 0.0002333636653375777}. Best is trial 3 with value: 0.6885330887142235.
[I 2026-01-26 17:38:41,588] Trial 4 finished with value: 0.6178156078382101 and pa

## 5. Visualizing Optimization Results
Optuna provides helpful visualizations to understand the search process.

In [ ]:
for model_name, study in studies.items():
    print(f"\n--- Optuna Plots for {model_name} ---")
    # Plot 1: Optimization History
    fig1 = plot_optimization_history(study)
    fig1.update_layout(title=f'{model_name} Optimization History')
    fig1.show()
    
    # Plot 2: Parameter Importances
    try:
        fig2 = plot_param_importances(study)
        fig2.update_layout(title=f'{model_name} Parameter Importances')
        fig2.show()
    except:
        print("Could not generate parameter importances plot (might need more completed trials).")

## 6. Final Validation with Best Hyperparameters
Finally, we take the best parameters found by Optuna for the top-performing model and run a full evaluation, including generating the detailed validation plots as requested.

In [ ]:
# Visualization functions from the original notebook (for the final evaluation)
def plot_aggregate_confusion(y_true, y_pred, le, title):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(le.classes_)))
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=le.classes_, yticklabels=le.classes_, cmap="Blues")
    plt.title(title)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.show()

# Find the best model among the studies
best_model_name = max(studies, key=lambda m: studies[m].best_value)
best_study = studies[best_model_name]
best_params = best_study.best_params

print(f"\n--- Running Final Validation on Best Model: {best_model_name} ---")
print("Using best parameters:", best_params)

# 1. Re-run pipeline with best data params
pipe = AdaptiveAccelPipeline(df_raw)
pipe.filter()
pipe.features()
df_r_best = pipe.resample(window=best_params['window'], thresh=best_params['thresh'])

feats = [c for c in df_r_best.columns if c not in ["subject", "local_ts", "label"] ]
le = LabelEncoder()
df_r_best["label_enc"] = le.fit_transform(df_r_best["label"])
df_r_best[feats] = StandardScaler().fit_transform(df_r_best[feats])
X, y, subjects = pipe.sequences(df_r_best, feats, "label_enc", steps=best_params['steps'])

# 2. Run LOSO with best model params
model_params = {'hidden_dim': best_params['hidden_dim'], 'lr': best_params['lr']}
y_true_final, y_pred_final = run_loso_for_tuning(X, y, subjects, le, best_model_name, model_params)

# 3. Display Final Validation Plot (Confusion Matrix)
print(f"\nFinal Overall Results for {best_model_name}:")
final_f1 = f1_score(y_true_final, y_pred_final, average='macro', zero_division=0)
final_acc = accuracy_score(y_true_final, y_pred_final)
print(f"  - Macro F1-Score: {final_f1:.4f}")
print(f"  - Accuracy: {final_acc:.4f}")

plot_aggregate_confusion(y_true_final, y_pred_final, le, f'Final Confusion Matrix for Best {best_model_name}')